# 02 · Visión de Thomas — guantes y manos

**Objetivo:** eliminar cualquier aspecto artificial de cuero/manos **sin modificar el bisturí**, su hoja, los adoquines, la sangre narrativa ni la disposición POV. Archivo original `web/assets/closeups/hands.jpg`, 1280×720.

El inpainting SOLO puede alterar las regiones dibujadas sobre los guantes; la fuente preserva su fondo pixel por pixel. Se procesan las dos manos en recortes independientes de 512×512 para la GPU T4. El original debe seguir reconocible: si aparecen dedos extra, falta de agarre, desaparece la hoja o cambia la iluminación de forma brusca, **rechaza el candidato**.

**Modelo de pesos abiertos, sin APIs de pago**; Colab GPU sí puede consumir unidades de Google AI Pro. No integra ni sube nada al repositorio.

In [ ]:
%pip -q install 'diffusers>=0.35,<0.40' 'transformers>=4.48,<5' 'accelerate>=1.0' 'safetensors>=0.5' 'Pillow>=10' 'numpy>=1.24' 'requests>=2.31'

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance, ImageOps, ImageChops
from IPython.display import display
import base64, io, json, math, hashlib, random, requests, shutil
REPO = "gabsvm/HorrorPlay"
COMMIT = "0681cfee36921ec2bd372f8ae8080a401f8bd63b"

def source(path):
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    r = requests.get(url, params={"ref": COMMIT}, headers={"Accept":"application/vnd.github+json"}, timeout=90)
    r.raise_for_status()
    data = r.json()
    if data.get("encoding") != "base64":
        raise RuntimeError(f"GitHub no devolvió el JPG/PNG en base64: {path}")
    b = base64.b64decode(data["content"])
    result = Image.open(io.BytesIO(b)).convert("RGB" if path.lower().endswith((".jpg", ".jpeg")) else "RGBA")
    return result, hashlib.sha256(b).hexdigest()

def font(size=20, bold=False):
    files = ["/usr/share/fonts/truetype/dejavu/DejaVuSerif-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
             "/usr/share/fonts/truetype/liberation2/LiberationSerif-Regular.ttf"]
    for f in files:
        if Path(f).exists(): return ImageFont.truetype(f, size)
    return ImageFont.load_default()

def compare(images, thumb=(640,360), labels=None):
    w,h=thumb
    sheet=Image.new('RGB',(w*len(images),h+36),(17,18,20))
    dr=ImageDraw.Draw(sheet)
    for i,im in enumerate(images):
        sheet.paste(im.convert('RGB').resize((w,h), Image.Resampling.LANCZOS),(i*w,36))
        dr.text((i*w+12,10), labels[i] if labels else str(i),fill=(245,241,229),font=font(15))
    return sheet

def finish(folder, manifest, outputs):
    manifest.update({"repo":REPO,"source_commit":COMMIT,"status":"CANDIDATO - No integrado ni testeado en Android"})
    (folder/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
    archive=shutil.make_archive(str(folder),'zip',folder)
    print('Exportado:', archive, 'archivos:', len(list(folder.iterdir())))
    from google.colab import files
    files.download(archive)

In [ ]:
out=Path('/content/horrorplay_hands_candidates');out.mkdir(parents=True,exist_ok=True)
original, sha = source('web/assets/closeups/hands.jpg')
assert original.size==(1280,720)
print('Fuente sha256:',sha)
display(original.resize((960,540)))

REGIONS={
  # ROI(x1,y1,x2,y2), polygon en coordenadas del lienzo ORIGINAL
  'left':{
    'crop':(100,70,612,582),
    'polygon':[(234,405),(224,330),(259,254),(328,190),(409,178),(462,217),
               (487,260),(516,297),(483,335),(496,376),(442,426),(323,482)]},
  'right':{
    'crop':(644,55,1156,567),
    'polygon':[(724,270),(749,217),(777,167),(819,151),(865,176),(892,154),
               (928,166),(951,192),(981,193),(1010,226),(995,295),(1002,373),
               (931,451),(833,425),(780,355)]}
}
mask_full=Image.new('L',original.size,0);dr=ImageDraw.Draw(mask_full)
for reg in REGIONS.values():dr.polygon(reg['polygon'],fill=255)
# Proteger el bisturí, especialmente hoja y borde del mango en la mano izquierda.
dr.polygon([(520,49),(568,31),(527,156),(477,247),(445,238),(493,148)],fill=0)
red=Image.new('RGBA',original.size,(220,30,35,0));red.putalpha(mask_full.point(lambda n:round(n*.45)))
display(Image.alpha_composite(original.convert('RGBA'),red).resize((960,540)))
print('Verifica que rojo=guantes; el bisturí y fondo deben estar fuera.')

## Generación T4 (primero una seed)

Si la máscara no te convence, modifica `REGIONS` antes de ejecutar. El modelo puede reinterpretar el guante y fallar en dedos. Mantener `strength` bajo reduce cambios anatómicos. La generación no está garantizada y necesita revisión manual.

In [ ]:
# Este bloque descarga pesos ABIERTOS (NO usa APIs de generación de pago).
# Google Colab consume unidades de GPU según el plan; probar primero un solo seed.
import torch
from diffusers import AutoPipelineForInpainting
assert torch.cuda.is_available(), 'Activa Entorno de ejecución > Cambiar tipo de entorno > T4 GPU'
MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-inpainting'
print('GPU:', torch.cuda.get_device_name(0))
pipe = AutoPipelineForInpainting.from_pretrained(MODEL_ID, torch_dtype=torch.float16,
                                               use_safetensors=False, variant='fp16')
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
print('Modelo disponible:', MODEL_ID)

In [ ]:
SEEDS=[1888]    # 2 manos × 1 seed; ampliar solo después de inspeccionar resultados
PROMPT=('extremely realistic first person view of worn Victorian dark brown leather gloves, '
        'anatomically correct gloved human hands, convincing articulated finger folds and leather grain, '
        'cinematic moody low candlelight, historical wool coat cuffs, filmic natural tactile realism, '
        'rain-wet Victorian cobblestone atmosphere, preserve original pose and objects')
NEG=('extra fingers, extra hands, missing fingers, plastic gloves, latex gloves, shiny artificial skin, '
     'wrong grip, misplaced knife, double knife, deformed anatomy, modern clothing, cartoon, CGI, text')

created=[]
for seed in SEEDS:
    composite=original.copy()
    for idx,(side,reg) in enumerate(REGIONS.items()):
        crop=reg['crop']
        mask=mask_full.crop(crop)
        assert mask.getbbox() and mask.size==(512,512)
        input_crop=composite.crop(crop)
        ai=pipe(prompt=PROMPT,negative_prompt=NEG,image=input_crop,
                mask_image=mask,width=512,height=512,
                strength=.35,num_inference_steps=28,guidance_scale=6,
                generator=torch.Generator(device='cpu').manual_seed(seed*10+idx)).images[0].convert('RGB')
        softmask=mask.filter(ImageFilter.GaussianBlur(8))
        composite.paste(Image.composite(ai,input_crop,softmask),crop)
    assert composite.size==original.size
    composite.save(out/f'hands_seed{seed}.png')
    composite.save(out/f'hands_seed{seed}.jpg',quality=89,optimize=True,subsampling=0)
    created.append(composite)
    print('Terminado:',seed)
preview=compare([original,*created],(640,360),['Original']+[f'Seed {s}' for s in SEEDS])
preview.save(out/'AB_hands_640.png');display(preview)
finish(out,{'sha256':sha,'model':MODEL_ID,'seeds':SEEDS,'regions':REGIONS,
           'preserve':'bisturí, sangre, adoquines y pose originales; rechazo si cambia anatomía'},[])